# 4DFlowNet Upgraded: K-Space Simulation & Sub-Pixel Ablations

This notebook demonstrates the upgraded 4DFlowNet-mini repository, reproducing the paper's key methodologies:

1. **Physically Realistic K-Space MRI Preprocessing**: Simulating VENC-based phase encoding, FFT downsampling, and complex Gaussian noise injection (Rayleigh noise in magnitude, uniform in background phase).
2. **Learned 3D Sub-Pixel Convolution (`PixelShuffle3d`)**: Replacing fixed trilinear upsampling with a learned convolution-based upsampler.
3. **2×2 Ablation Sweep**: Evaluating `{Spatial Gaussian vs K-Space Noise} × {Trilinear vs Sub-Pixel Upsampling}` to measure impact on Peak Velocity and Net Flow errors.

## 0. Colab Environment Setup

Set runtime to **GPU** (Runtime -> Change runtime type -> T4 GPU or higher). Then clone the repository and install requirements if on Colab.

In [ ]:
# Clone and set up directory structure if running in Google Colab
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/eliasubz/4D-FlowNet.git
    %cd 4D-FlowNet
    !pip install -r requirements.txt
else:
    print("Running locally. Current working directory:")
    !pwd

## 1. Imports and Initialization

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import pandas as pd

from src.dataset import SyntheticFlowDataset, upsample_lr
from src.mri_simulation import simulate_mri_acquisition
from src.model import FourDFlowNet
from src.losses import four_d_flow_loss
from src.metrics import endpoint_error, peak_velocity_error, flow_rate_error, divergence_l1
from src.visualize import make_interactive_3d_flow_figure

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

## 2. Visualize Physical MRI K-Space Simulation vs Spatial Noise

Here we compare the simple spatial Gaussian noise against the physics-based k-space simulation. Look at how k-space noise creates correlated noise patterns and Rayleigh-distributed magnitude textures in non-flow regions.

In [ ]:
# Load a single synthetic datapoint with spatial noise
ds_spatial = SyntheticFlowDataset(samples=1, use_kspace_noise=False, noise_std=0.08, seed=42)
lr_spatial, hr = ds_spatial[0]

# Load the same datapoint with k-space noise simulation
ds_kspace = SyntheticFlowDataset(samples=1, use_kspace_noise=True, snr_range=(14.0, 14.0), seed=42)
lr_kspace, _ = ds_kspace[0]

# Speeds
hr_speed = torch.linalg.vector_norm(hr, dim=0)
spatial_speed = torch.linalg.vector_norm(lr_spatial[:3], dim=0)
kspace_speed = torch.linalg.vector_norm(lr_kspace[:3], dim=0)

z_lr = lr_spatial.shape[1] // 2
z_hr = hr.shape[1] // 2

fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)

# Ground Truth
im0 = axes[0, 0].imshow(hr_speed[z_hr], cmap='viridis', origin='lower')
axes[0, 0].set_title("HR Ground Truth Speed")
fig.colorbar(im0, ax=axes[0, 0])

# Spatial Noisy Input
im1 = axes[0, 1].imshow(spatial_speed[z_lr], cmap='viridis', origin='lower')
axes[0, 1].set_title("Spatial Noisy Input Speed (Gaussian)")
fig.colorbar(im1, ax=axes[0, 1])

# K-Space Noisy Input
im2 = axes[0, 2].imshow(kspace_speed[z_lr], cmap='viridis', origin='lower')
axes[0, 2].set_title("K-Space Noisy Input Speed (Rayleigh)")
fig.colorbar(im2, ax=axes[0, 2])

# Spatial Magnitude
im3 = axes[1, 1].imshow(lr_spatial[3, z_lr], cmap='bone', origin='lower')
axes[1, 1].set_title("Spatial Magnitude Channel")
fig.colorbar(im3, ax=axes[1, 1])

# K-Space Magnitude
im4 = axes[1, 2].imshow(lr_kspace[3, z_lr], cmap='bone', origin='lower')
axes[1, 2].set_title("K-Space Magnitude (Rayleigh Noise Texture)")
fig.colorbar(im4, ax=axes[1, 2])

# Hide unused axes[1, 0]
axes[1, 0].axis('off')

plt.show()

## 3. Compare Upsampling Architectures: Trilinear vs Sub-Pixel Convolution

Compare parameters and shapes of `FourDFlowNet` under both upsampling modes.

In [ ]:
model_trilinear = FourDFlowNet(width=32, lr_blocks=4, hr_blocks=2, upsample_mode='trilinear')
model_subpixel = FourDFlowNet(width=32, lr_blocks=4, hr_blocks=2, upsample_mode='subpixel')

print(f"Trilinear Model Parameters: {sum(p.numel() for p in model_trilinear.parameters()):,}")
print(f"Sub-Pixel Model Parameters: {sum(p.numel() for p in model_subpixel.parameters()):,}")

# Run dummy batch through both
dummy_input = torch.randn(2, 6, 16, 16, 16)
out_trilinear = model_trilinear(dummy_input)
out_subpixel = model_subpixel(dummy_input)

print(f"Input shape: {dummy_input.shape}")
print(f"Trilinear output shape: {out_trilinear.shape}")
print(f"Sub-Pixel output shape: {out_subpixel.shape}")

## 4. Define Training and Evaluation Sweeps

This helper function runs an individual experiment configuration and collects epoch-level history.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    totals = {
        'mae': 0.0, 'interp_mae': 0.0,
        'epe': 0.0, 'interp_epe': 0.0,
        'peak_err': 0.0, 'interp_peak_err': 0.0,
        'flow_err': 0.0, 'interp_flow_err': 0.0
    }
    batches = 0
    for lr, hr in loader:
        lr, hr = lr.to(device), hr.to(device)
        pred = model(lr)
        interp = upsample_lr(lr, hr.shape[-1])

        totals['mae'] += (pred - hr).abs().mean().item()
        totals['interp_mae'] += (interp - hr).abs().mean().item()
        totals['epe'] += endpoint_error(pred, hr).item()
        totals['interp_epe'] += endpoint_error(interp, hr).item()
        totals['peak_err'] += peak_velocity_error(pred, hr).item()
        totals['interp_peak_err'] += peak_velocity_error(interp, hr).item()
        totals['flow_err'] += flow_rate_error(pred, hr).item()
        totals['interp_flow_err'] += flow_rate_error(interp, hr).item()
        batches += 1

    return {k: v / batches for k, v in totals.items()}

def run_experiment(name, use_kspace_noise, upsample_mode, epochs=15, batch_size=4, train_samples=512, val_samples=128):
    print(f"\n{'='*40}\nStarting Experiment: {name}\nk-space={use_kspace_noise}, upsampling={upsample_mode}\n{'='*40}")
    
    dataset_kwargs = dict(noise_std=0.03)
    if use_kspace_noise:
        dataset_kwargs.update(use_kspace_noise=True, snr_range=(14.0, 17.0))
        
    train_data = SyntheticFlowDataset(train_samples, seed=10, **dataset_kwargs)
    val_data = SyntheticFlowDataset(val_samples, seed=10000, **dataset_kwargs)
    
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    
    model = FourDFlowNet(width=24, lr_blocks=4, hr_blocks=2, upsample_mode=upsample_mode).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type=='cuda')
    
    history = []
    
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for lr, hr in train_loader:
            lr, hr = lr.to(device), hr.to(device)
            optimizer.zero_grad(set_to_none=True)
            
            with torch.amp.autocast('cuda', enabled=device.type=='cuda'):
                pred = model(lr)
                loss = four_d_flow_loss(pred, hr, gradient_weight=1e-3)
                
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            
        scheduler.step()
        
        epoch_loss = running_loss / len(train_loader)
        metrics = evaluate(model, val_loader, device)
        metrics['train_loss'] = epoch_loss
        metrics['epoch'] = epoch
        history.append(metrics)
        
        if epoch % 5 == 0 or epoch == epochs:
            print(f"Epoch {epoch}/{epochs} - loss: {epoch_loss:.5f} - MAE: {metrics['mae']:.5f} (interp: {metrics['interp_mae']:.5f}) - Peak Err: {100*metrics['peak_err']:.1f}% - Flow Err: {100*metrics['flow_err']:.1f}%")
            
    return history, model

## 5. Run the 2x2 Ablation Sweep

Let's execute all 4 configurations to compare their performance under equal training budgets.

In [ ]:
configs = [
    ("baseline_spatial", False, "trilinear"),
    ("kspace_only",       True,  "trilinear"),
    ("subpixel_only",     False, "subpixel"),
    ("full_paper",        True,  "subpixel")
]

results = {}
trained_models = {}
for name, use_kspace, up_mode in configs:
    history, model = run_experiment(name, use_kspace, up_mode, epochs=15, batch_size=8, train_samples=1024)
    results[name] = history
    trained_models[name] = model

## 6. Analyze Results & Compare Metrics

Plot the training curves and compare the final metric breakdown in a quantitative table.

In [ ]:
# Print comparison table of final metrics
summary_data = []
for name, history in results.items():
    final = history[-1]
    summary_data.append({
        'Experiment': name,
        'Train Loss': f"{final['train_loss']:.5f}",
        'Model MAE': f"{final['mae']:.5f}",
        'Interp MAE': f"{final['interp_mae']:.5f}",
        'Peak Err %': f"{100*final['peak_err']:.2f}%",
        'Interp Peak %': f"{100*final['interp_peak_err']:.2f}%",
        'Flow Err %': f"{100*final['flow_err']:.2f}%",
        'Interp Flow %': f"{100*final['interp_flow_err']:.2f}%"
    })
df = pd.DataFrame(summary_data)
print(df.to_markdown(index=False))

# Plot MAE over epochs
plt.figure(figsize=(10, 6))
for name, history in results.items():
    epochs = [h['epoch'] for h in history]
    maes = [h['mae'] for h in history]
    plt.plot(epochs, maes, label=name, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Validation MAE')
plt.title('Ablation Analysis: Epoch vs Validation MAE')
plt.grid(True)
plt.legend()
plt.show()

## 7. Interactive 3D Flow Visualization of Reconstruction

Use Plotly to visualize the 3D velocity vectors inside the vessel from the `full_paper` model prediction.

In [ ]:
ds_preview = SyntheticFlowDataset(samples=1, use_kspace_noise=True, seed=999)
lr, hr = ds_preview[0]
lr_batch = lr.unsqueeze(0).to(device)

model_paper = trained_models['full_paper']
model_paper.eval()
with torch.no_grad():
    pred_vel = model_paper(lr_batch).squeeze(0).cpu()

fig3d = make_interactive_3d_flow_figure(
    pred_vel,
    wall_threshold=0.03,
    vector_threshold=0.10,
    vector_step=3,
    arrow_scale=0.15
)
fig3d.show()